In [1]:
import pandas as pd
import numpy as np

# Load dataset bersih yang sudah kita siapkan
df = pd.read_csv("../data/kost_putri_cleaned.csv")

print(f"Total data kos yang siap dianalisis: {len(df)} baris")
df[['Nama Kos', 'harga_min', 'harga_max', 'has_ac', 'has_km_dalam', 'has_wifi']].head()

Total data kos yang siap dianalisis: 63 baris


,Nama Kos,harga_min,harga_max,has_ac,has_km_dalam,has_wifi
0,Kost Putri Asri,750000.0,1200000.0,False,True,True
1,Kost Putri Pak Seno,800000.0,1000000.0,True,False,True
2,Kost Perum APH Seturan A.24,600000.0,700000.0,False,False,True
3,Graha Seturan Kost Putri,850000.0,850000.0,False,True,True
4,Kost Diajeng,900000.0,900000.0,False,True,True


In [2]:
def calculate_price_score(harga_kos, user_budget, tolerance=250000):
    """
    Menghitung skor kesesuaian harga (0.0 sampai 1.0)
    """
    if pd.isna(harga_kos):
        # Jika harga belum tercatat, berikan skor default netral
        return 0.5
    
    # Jika harga di dalam budget, skor sempurna
    if harga_kos <= user_budget:
        return 1.0
    
    # Jika melebihi budget, hitung penalti selisih
    diff = harga_kos - user_budget
    if diff <= tolerance:
        # Penalti linier: semakin dekat dengan toleransi, skor mendekati 0
        return max(0.0, 1.0 - (diff / tolerance))
    else:
        return 0.0

# Pengujian sederhana
print("Tes Budget 800.000:")
print(f"Harga 700.000 (di bawah budget) : {calculate_price_score(700000, 800000)}")
print(f"Harga 800.000 (pas budget)      : {calculate_price_score(800000, 800000)}")
print(f"Harga 900.000 (selisih 100rb)   : {calculate_price_score(900000, 800000):.2f}")
print(f"Harga 1.200.000 (lewat toleransi): {calculate_price_score(1200000, 800000)}")

Tes Budget 800.000:
Harga 700.000 (di bawah budget) : 1.0
Harga 800.000 (pas budget)      : 1.0
Harga 900.000 (selisih 100rb)   : 0.60
Harga 1.200.000 (lewat toleransi): 0.0


In [3]:
def calculate_facility_score(row, user_weights):
    """
    Menghitung skor kecocokan fasilitas berbasis bobot preferensi pengguna.
    user_weights: dict berisi {nama_fitur: bobot(0-3)}
    Contoh: {'has_ac': 3, 'has_wifi': 2, 'has_km_dalam': 3, 'has_listrik': 1}
    """
    total_weight = 0
    earned_score = 0
    
    for feature, weight in user_weights.items():
        if weight > 0:
            total_weight += weight
            # Jika kos memiliki fasilitas tersebut (bernilai True)
            if row.get(feature) is True:
                earned_score += weight
                
    if total_weight == 0:
        return 1.0  # Jika pengguna tidak mementingkan fasilitas apapun
        
    return earned_score / total_weight

In [4]:
def calculate_location_score(row, preferred_locations):
    """
    Menghitung kecocokan lokasi (0.0 sampai 1.0)
    preferred_locations: list kata kunci area, contoh: ['kledokan', 'seturan']
    """
    if not preferred_locations:
        return 1.0  # Pengguna fleksibel soal lokasi
        
    text = (str(row.get('Alamat', '')) + " " + str(row.get('Nama Kos', ''))).lower()
    
    matches = sum(1 for loc in preferred_locations if loc.lower() in text)
    return min(1.0, matches / len(preferred_locations))

In [5]:
def recommend_kost(
    df_data,
    user_budget=1000000,
    budget_tolerance=250000,
    facility_weights=None,
    preferred_locations=None,
    weight_components={'price': 0.40, 'facility': 0.45, 'location': 0.15},
    top_n=5
):
    """
    Menghitung skor kecocokan total (Content-Based Matching) dan mengembalikan Top-N kos terbaik.
    """
    if facility_weights is None:
        facility_weights = {
            'has_ac': 2,
            'has_km_dalam': 3,
            'has_wifi': 2,
            'has_listrik': 1
        }
    
    if preferred_locations is None:
        preferred_locations = []
        
    results = []
    
    for idx, row in df_data.iterrows():
        # 1. Skor Harga (menggunakan harga_min sebagai patokan dasar tipe termurah)
        p_score = calculate_price_score(row['harga_min'], user_budget, budget_tolerance)
        
        # 2. Skor Fasilitas
        f_score = calculate_facility_score(row, facility_weights)
        
        # 3. Skor Lokasi
        l_score = calculate_location_score(row, preferred_locations)
        
        # 4. Total Weighted Score (Skala 0 - 100%)
        total_score = (
            (p_score * weight_components['price']) +
            (f_score * weight_components['facility']) +
            (l_score * weight_components['location'])
        ) * 100
        
        results.append({
            'No.': row.get('No.'),
            'Nama Kos': row['Nama Kos'],
            'Match Score (%)': round(total_score, 1),
            'Biaya': row['Biaya Perbulan/semester'],
            'harga_min': row['harga_min'],
            'Alamat': row['Alamat'],
            'Fasilitas': row['Fasilitas Kos'],
            'wa_url': row.get('wa_url'),
            'maps_url': row.get('maps_url'),
            'Skor Harga': round(p_score * 100, 1),
            'Skor Fasilitas': round(f_score * 100, 1),
            'Skor Lokasi': round(l_score * 100, 1)
        })
        
    results_df = pd.DataFrame(results)
    
    # Urutkan berdasarkan Match Score tertinggi
    results_df = results_df.sort_values(by='Match Score (%)', ascending=False)
    
    return results_df.head(top_n)

In [6]:
# SKENARIO PENGUJIAN:
# Mahasiswi baru mencari kos:
# - Budget: Rp850.000/bulan (toleransi selisih max Rp200.000)
# - Prioritas Fasilitas: Kamar Mandi Dalam WAJIB (bobot 3), Wi-Fi PENTING (bobot 2), AC TIDAK BUTUH (bobot 0), Listrik BIASA (bobot 1)
# - Lokasi: Diutamakan area 'Kledokan'

profil_user = {
    'user_budget': 850000,
    'budget_tolerance': 200000,
    'facility_weights': {
        'has_km_dalam': 3,
        'has_wifi': 2,
        'has_ac': 0,
        'has_listrik': 1
    },
    'preferred_locations': ['kledokan'],
    'top_n': 5
}

# Jalankan rekomendasi
top_rekomendasi = recommend_kost(
    df,
    user_budget=profil_user['user_budget'],
    budget_tolerance=profil_user['budget_tolerance'],
    facility_weights=profil_user['facility_weights'],
    preferred_locations=profil_user['preferred_locations'],
    top_n=profil_user['top_n']
)

# Tampilkan hasil
top_rekomendasi[['Nama Kos', 'Match Score (%)', 'Biaya', 'Skor Harga', 'Skor Fasilitas', 'Skor Lokasi']]

,Nama Kos,Match Score (%),Biaya,Skor Harga,Skor Fasilitas,Skor Lokasi
3,Graha Seturan Kost Putri,100.0,Rp850.000,100.0,100.0,100.0
46,Kost Putri Tamara Hotel MM,100.0,Rp795.000/bulan,100.0,100.0,100.0
47,Kost Putri Tamara Lala,100.0,Rp750.000/bulan,100.0,100.0,100.0
15,Kost Grahanadhi,92.5,Rp800.000/bulan\ndibayar pertigabulan,100.0,83.3,100.0
4,Kost Diajeng,90.0,Rp900.000/bulan\ndibayar pertigabulan,75.0,100.0,100.0
